<a href="https://colab.research.google.com/github/JabulaniMcineka/MyProjects/blob/main/sports_data_pipeline_spark_sqs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install PySpark
!pip install pyspark -q

# Test PySpark works
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SportsDataPipeline") \
    .getOrCreate()

print("PySpark is ready!")
print(f"Spark version: {spark.version}")

PySpark is ready!
Spark version: 4.0.2


In [5]:
!pip install boto3 pyspark requests -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.9 MB/s eta 0:00:00


In [6]:
import boto3
import json
import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp
from google.colab import userdata
from datetime import datetime

# ---- CREDENTIALS ----
AWS_ACCESS_KEY = userdata.get('AWS_ACCESS_KEY')
AWS_SECRET_KEY = userdata.get('AWS_SECRET_KEY')

# ---- STEP 1: FETCH SPORTS DATA ----
print(" Fetching sports data from API...")
url = "https://www.thesportsdb.com/api/v1/json/3/eventspastleague.php"
params = {"id": "4328"}
response = requests.get(url, params=params)
events = response.json()['events']
print(f" Fetched {len(events)} events!")

# ---- STEP 2: SEND TO SQS ----
print(" Sending data to SQS queue...")
sqs = boto3.client(
    'sqs',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name='us-east-1'
)

# Get queue URL
queue_url = sqs.get_queue_url(QueueName='sports-data-queue')['QueueUrl']

# Send each event as a message
for event in events:
    sqs.send_message(
        QueueUrl=queue_url,
        MessageBody=json.dumps({
            'event_id': event['idEvent'],
            'date': event['strTimestamp'],
            'home_team': event['strEvent'].split(' vs ')[0],
            'away_team': event['strEvent'].split(' vs ')[1],
            'league': event['strLeague'],
            'sport': event['strSport']
        })
    )

print(f" {len(events)} messages sent to SQS!")

# ---- STEP 3: READ FROM SQS ----
print(" Reading messages from SQS...")
messages = []
while True:
    response = sqs.receive_message(
        QueueUrl=queue_url,
        MaxNumberOfMessages=10,
        WaitTimeSeconds=2
    )
    if 'Messages' not in response:
        break
    for msg in response['Messages']:
        messages.append(json.loads(msg['Body']))
        sqs.delete_message(
            QueueUrl=queue_url,
            ReceiptHandle=msg['ReceiptHandle']
        )

print(f" Received {len(messages)} messages from SQS!")

# ---- STEP 4: PROCESS WITH PYSPARK ----
print(" Processing data with PySpark...")
spark = SparkSession.builder \
    .appName("SportsDataPipeline") \
    .getOrCreate()

df_spark = spark.createDataFrame(messages)
df_spark = df_spark.withColumn("date", to_timestamp(col("date")))

print(" PySpark processing complete!")
df_spark.show(5)

# ---- STEP 5: SAVE TO S3 ----
print(" Saving to S3...")
df_pandas = df_spark.toPandas()

s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name='us-east-1'
)

filename = f"spark_processed_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

s3.put_object(
    Bucket="sports-data-transformed-7977-9545-4172",
    Key=f"spark/{filename}",
    Body=df_pandas.to_json(orient='records'),
    ContentType='application/json'
)

print(f"Data saved to S3: spark/{filename}")
print("\n FULL PIPELINE COMPLETE!")
print("Sports API → SQS → PySpark → S3")

 Fetching sports data from API...
 Fetched 15 events!
 Sending data to SQS queue...
 15 messages sent to SQS!
 Reading messages from SQS...
 Received 15 messages from SQS!
 Processing data with PySpark...
 PySpark processing complete!
+-----------------+-------------------+--------+----------------+----------------+------+
|        away_team|               date|event_id|       home_team|          league| sport|
+-----------------+-------------------+--------+----------------+----------------+------+
|     Cardiff City|2026-03-14 15:00:00| 2275077|     Exeter City|English League 1|Soccer|
| Bolton Wanderers|2026-03-14 12:30:00| 2275073|Rotherham United|English League 1|Soccer|
|Huddersfield Town|2026-03-14 15:00:00| 2275071|       Port Vale|English League 1|Soccer|
|         Barnsley|2026-03-14 15:00:00| 2275069|  Mansfield Town|English League 1|Soccer|
| Stockport County|2026-03-14 12:30:00| 2275068|    Lincoln City|English League 1|Soccer|
+-----------------+-------------------+------